In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from catboost import CatBoostRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real

import os
import json
import joblib
import matplotlib.pyplot as plt

In [ ]:
# Data
df = pd.read_csv("StudentsPerformance_modified.csv")

# X: on EXCLUT reading/writing/math + la cible créée
X = df.drop(columns=["reading score", "writing score", "math score", "language mean score"])
y_lang = df["language mean score"]

# Categorical indices for CatBoost
cat_cols = X.select_dtypes(include="object").columns.tolist()
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X, y_lang, test_size=0.2, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

base_lang = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

search_space = {
    "depth": Integer(3, 10),
    "learning_rate": Real(1e-3, 0.3, prior="log-uniform"),
    "l2_leaf_reg": Real(1.0, 20.0),
    "iterations": Integer(300, 2000),
    "subsample": Real(0.5, 1.0),
    "random_strength": Real(0.0, 2.0),
}

opt_lang = BayesSearchCV(
    estimator=base_lang,
    search_spaces=search_space,
    n_iter=35,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=0
)

opt_lang.fit(X_train_l, y_train_l, cat_features=cat_idx)

best_model_lang = opt_lang.best_estimator_
best_params_lang = opt_lang.best_params_
best_cv_r2_lang = float(opt_lang.best_score_)

print(" Bayes opt OK (language mean score). Best CV R2:", best_cv_r2_lang)
print("Best params:", best_params_lang)

best_model_lang.fit(X_train_l, y_train_l, cat_features=cat_idx)

pred_lang = best_model_lang.predict(X_test_l)

rmse_val = float(np.sqrt(mean_squared_error(y_test_l, pred_lang)))

lang_metrics = {
    "cv_best_r2": best_cv_r2_lang,
    "r2_test": float(r2_score(y_test_l, pred_lang)),
    "mae_test": float(mean_absolute_error(y_test_l, pred_lang)),
    "rmse_test": rmse_val,
    "y_test_mean": float(np.mean(y_test_l)),
}

print("\n language mean score test metrics:", lang_metrics)

In [ ]:
os.makedirs("artifacts", exist_ok=True)

# ---------- Graphs ----------
plt.scatter(y_test_l, pred_lang, alpha=0.6)
plt.xlabel("True score")
plt.ylabel("Predicted score")
plt.title("Language Mean Score — True vs Pred")
mn = min(y_test_l.min(), pred_lang.min())
mx = max(y_test_l.max(), pred_lang.max())
plt.plot([mn, mx], [mn, mx])
plt.show()

resid = y_test_l - pred_lang
plt.figure()
plt.scatter(pred_lang, resid, alpha=0.6)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Language Mean Score — Residuals vs Pred")
plt.show()

plt.figure()
plt.bar(X.columns, best_model_lang.feature_importances_)
plt.xticks(rotation=90)
plt.title("Feature Importance — language mean score")
plt.show()

# ---------- Export model ----------
joblib.dump(best_model_lang, "artifacts/catboost_language_mean_score.joblib")

# ---------- Model card ----------
model_card = f"""# Model Card — CatBoost Language Mean Score Predictor

## Overview
- **Task**: Regression (predict 'language mean score')
- **Target definition**: '(reading score + writing score) / 2'
- **Model**: CatBoostRegressor
- **Excluded features**: reading score, writing score, math score, language mean score
- **Categorical features**: {cat_cols}

## Data & preprocessing
- Dataset: 'StudentsPerformance_modified.csv'
- Categorical handling: CatBoost native categorical support (no one-hot)

## Training procedure
- Train/Test split: 80/20 (random_state=42)
- Cross-validation: 5-fold KFold (shuffle=True)
- Hyperparameter tuning: Bayesian Optimization (BayesSearchCV, scoring=R²)

## Performance (Test set)
- **Best CV R²**: {lang_metrics["cv_best_r2"]:.4f}
- **Test R²**: {lang_metrics["r2_test"]:.4f}
- **MAE**: {lang_metrics["mae_test"]:.4f}
- **RMSE**: {lang_metrics["rmse_test"]:.4f}
- **Mean target**: {lang_metrics["y_test_mean"]:.4f}

## Best hyperparameters
'''json
{json.dumps(best_params_lang, indent=2, ensure_ascii=False)}